# Calculadoras de Distribuições Discretas

Este notebook contém calculadoras interativas para as principais distribuições de probabilidade discretas.

## 💡 Aproximações Possíveis

| De (Original) | Para (Aproximada) | Critério |
| :--- | :--- | :--- |
| **Hipergeométrica** | **Binomial** | $M \geq 10N$ |
| **Binomial** | **Poisson** | $N \geq 20$ e ($Np \leq 7$ ou $Nq \leq 7$) |
| **Binomial** | **Normal** | $N \geq 20, Np > 7, Nq > 7$ |
| **Poisson** | **Normal** | $\lambda > 10$ |

In [ ]:
import scipy.stats as stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt


## 1. Distribuição Binomial

A distribuição binomial modela o número de sucessos em n tentativas independentes, cada uma com probabilidade p de sucesso.

**Parâmetros:**
- n: número de tentativas
- p: probabilidade de sucesso em cada tentativa
- X: número de sucessos

In [ ]:
import scipy.stats as stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')

# Flag para prevenir chamadas duplicadas
_updating_binom = False

# Criar widgets
modo = widgets.ToggleButtons(
    options=['Direto (k → P)', 'Inverso (P → k)'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

n_widget = widgets.IntText(value=20, description='n (tentativas):', style={'description_width': 'initial'})
p_widget = widgets.FloatText(value=0.3, description='p (probabilidade):', style={'description_width': 'initial'})
k_widget = widgets.IntText(value=6, description='k (sucessos):', style={'description_width': 'initial'})
a_widget = widgets.IntText(value=4, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget = widgets.IntText(value=10, description='b (limite sup.):', style={'description_width': 'initial'})
prob_widget = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type = widgets.Dropdown(
    options=[
        ('P(X = k)', 'pmf'), 
        ('P(X ≤ k)', 'cdf'), 
        ('P(X ≥ k)', 'sf'), 
        ('P(X > k)', 'gt'), 
        ('P(X < k)', 'lt'),
        ('P(a ≤ X ≤ b)', 'interval')
    ],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_inv = widgets.Dropdown(
    options=[('P(X ≤ k) ≥ p', 'cdf'), ('P(X ≥ k) ≤ p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

show_graph = widgets.Checkbox(value=True, description='Mostrar gráfico')
output = widgets.Output()
input_container = widgets.VBox()

def calcular_binomial(change=None):
    global _updating_binom
    if _updating_binom:
        return
    _updating_binom = True
    
    try:
        with output:
            clear_output(wait=True)
            try:
                n = n_widget.value
                p = p_widget.value
                q = 1 - p
                
                if n < 0 or p < 0 or p > 1:
                    print("❌ Valores inválidos! Certifique-se: n ≥ 0, 0 ≤ p ≤ 1")
                    return
                
                dist = stats.binom(n, p)
                
                # Modo Direto
                if modo.value == 'Direto (k → P)':
                    if calc_type.value == 'interval':
                        a = a_widget.value
                        b = b_widget.value
                        
                        if a < 0 or b > n or a > b:
                            print(f"❌ Valores inválidos! 0 ≤ a ≤ b ≤ {n}")
                            return
                        
                        prob = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                        print(f"P({a} ≤ X ≤ {b}) = {prob:.6f}")
                        print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                        print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 0 else 0:.6f}")
                        
                        if show_graph.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(0, n+1)
                            pmf_vals = dist.pmf(x_vals)
                            
                            colors = ['red' if a <= x <= b else 'steelblue' for x in x_vals]
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                            ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Sucessos (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Binomial: n={n}, p={p}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        k = k_widget.value
                        if k < 0 or k > n:
                            print(f"❌ Valor inválido! k deve estar entre 0 e {n}")
                            return
                        
                        if calc_type.value == 'pmf':
                            result = dist.pmf(k)
                            print(f"P(X = {k}) = {result:.6f}")
                        elif calc_type.value == 'cdf':
                            result = dist.cdf(k)
                            print(f"P(X ≤ {k}) = {result:.6f}")
                        elif calc_type.value == 'sf':
                            result = dist.sf(k-1)
                            print(f"P(X ≥ {k}) = {result:.6f}")
                        elif calc_type.value == 'gt':
                            result = dist.sf(k)
                            print(f"P(X > {k}) = {result:.6f}")
                        elif calc_type.value == 'lt':
                            result = dist.cdf(k-1) if k > 0 else 0
                            print(f"P(X < {k}) = {result:.6f}")
                        
                        print(f"\nEsperança E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        # Verificar aproximações
                        print("\n" + "="*50)
                        print("🔔 AVISOS DE APROXIMAÇÃO:")
                        
                        if n >= 20 and (n*p <= 7 or n*q <= 7):
                            lam = n * p
                            print(f"\n✅ Pode aproximar para POISSON com λ = {lam:.4f}")
                            print(f"   Critério: N={n} ≥ 20 e Np={n*p:.2f} ≤ 7 ou Nq={n*q:.2f} ≤ 7")
                        
                        if n >= 20 and n*p > 7 and n*q > 7:
                            mu = n * p
                            sigma = np.sqrt(n * p * q)
                            print(f"\n✅ Pode aproximar para NORMAL com μ = {mu:.4f}, σ = {sigma:.4f}")
                            print(f"   Critério: N={n} ≥ 20, Np={n*p:.2f} > 7, Nq={n*q:.2f} > 7")
                        
                        if n < 20 or (n*p <= 7 and n*q <= 7):
                            print("\n⚠️  Nenhuma aproximação recomendada. Use a distribuição Binomial.")
                        
                        if show_graph.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(0, n+1)
                            pmf_vals = dist.pmf(x_vals)
                            
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color='steelblue', edgecolor='black')
                            ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Sucessos (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Binomial: n={n}, p={p}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    prob = prob_widget.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    if calc_type_inv.value == 'cdf':
                        k = dist.ppf(prob)
                        print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                        print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                    else:  # sf
                        k = dist.isf(prob)
                        print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                        print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        if int(k) >= 1:
                            print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
                    
                    print(f"\nEsperança E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    print(f"Desvio Padrão σ = {dist.std():.4f}")
                    
                    if show_graph.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.arange(0, n+1)
                        pmf_vals = dist.pmf(x_vals)
                        
                        ax.bar(x_vals, pmf_vals, alpha=0.7, color='steelblue', edgecolor='black')
                        ax.axvline(int(k), color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                        ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        ax.set_xlabel('Número de Sucessos (k)', fontsize=12)
                        ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                        ax.set_title(f'Distribuição Binomial: n={n}, p={p}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_binom = False

def atualizar_interface(change=None):
    global _updating_binom
    if _updating_binom:
        return
    _updating_binom = True
    
    try:
        if modo.value == 'Direto (k → P)':
            if calc_type.value == 'interval':
                input_container.children = [calc_type, a_widget, b_widget]
            else:
                input_container.children = [calc_type, k_widget]
        else:
            input_container.children = [calc_type_inv, prob_widget]
        calcular_binomial()
    finally:
        _updating_binom = False

# Registrar observadores apenas uma vez
if not hasattr(modo, '_handler_registered'):
    modo.observe(atualizar_interface, 'value')
    calc_type.observe(atualizar_interface, 'value')
    n_widget.observe(calcular_binomial, 'value')
    p_widget.observe(calcular_binomial, 'value')
    k_widget.observe(calcular_binomial, 'value')
    a_widget.observe(calcular_binomial, 'value')
    b_widget.observe(calcular_binomial, 'value')
    prob_widget.observe(calcular_binomial, 'value')
    calc_type_inv.observe(calcular_binomial, 'value')
    show_graph.observe(calcular_binomial, 'value')
    
    modo._handler_registered = True
    calc_type._handler_registered = True
    n_widget._handler_registered = True
    p_widget._handler_registered = True
    k_widget._handler_registered = True
    a_widget._handler_registered = True
    b_widget._handler_registered = True
    prob_widget._handler_registered = True
    calc_type_inv._handler_registered = True
    show_graph._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Binomial</h3>"),
    modo,
    n_widget, 
    p_widget, 
    input_container,
    show_graph, 
    output
]))

atualizar_interface()

## 2. Distribuição de Poisson

A distribuição de Poisson modela o número de eventos que ocorrem em um intervalo fixo de tempo ou espaço.

**Parâmetros:**
- λ (lambda): taxa média de ocorrência
- X: número de eventos

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_poisson = False

# Criar widgets
modo_poisson = widgets.ToggleButtons(
    options=['Direto (k → P)', 'Inverso (P → k)'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

lambda_widget = widgets.FloatText(value=5.0, description='λ (taxa média):', style={'description_width': 'initial'})
k_poisson = widgets.IntText(value=7, description='k (eventos):', style={'description_width': 'initial'})
a_poisson = widgets.IntText(value=3, description='a (limite inf.):', style={'description_width': 'initial'})
b_poisson = widgets.IntText(value=8, description='b (limite sup.):', style={'description_width': 'initial'})
prob_widget_poisson = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_poisson = widgets.Dropdown(
    options=[
        ('P(X = k)', 'pmf'), 
        ('P(X ≤ k)', 'cdf'), 
        ('P(X ≥ k)', 'sf'), 
        ('P(X > k)', 'gt'), 
        ('P(X < k)', 'lt'),
        ('P(a ≤ X ≤ b)', 'interval')
    ],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_poisson_inv = widgets.Dropdown(
    options=[('P(X ≤ k) ≥ p', 'cdf'), ('P(X ≥ k) ≤ p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

show_graph_poisson = widgets.Checkbox(value=True, description='Mostrar gráfico')
output_poisson = widgets.Output()
input_container_poisson = widgets.VBox()

def calcular_poisson(change=None):
    global _updating_poisson
    if _updating_poisson:
        return
    _updating_poisson = True
    
    try:
        with output_poisson:
            clear_output(wait=True)
            try:
                lam = lambda_widget.value
                
                if lam <= 0:
                    print("❌ Valores inválidos! λ > 0")
                    return
                
                dist = stats.poisson(lam)
                
                # Modo Direto
                if modo_poisson.value == 'Direto (k → P)':
                    if calc_type_poisson.value == 'interval':
                        a = a_poisson.value
                        b = b_poisson.value
                        
                        if a < 0 or a > b:
                            print("❌ Valores inválidos! 0 ≤ a ≤ b")
                            return
                        
                        prob = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                        print(f"P({a} ≤ X ≤ {b}) = {prob:.6f}")
                        print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                        print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 0 else 0:.6f}")
                        
                        if show_graph_poisson.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(0, int(lam + 5*np.sqrt(lam)))
                            pmf_vals = dist.pmf(x_vals)
                            
                            colors = ['red' if a <= x <= b else 'coral' for x in x_vals]
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                            ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                            ax.axvline(lam, color='green', linestyle='--', linewidth=2, label=f'λ = {lam:.2f}')
                            
                            ax.set_xlabel('Número de Eventos (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição de Poisson: λ={lam}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        k = k_poisson.value
                        if k < 0:
                            print("❌ Valor inválido! k ≥ 0")
                            return
                        
                        if calc_type_poisson.value == 'pmf':
                            result = dist.pmf(k)
                            print(f"P(X = {k}) = {result:.6f}")
                        elif calc_type_poisson.value == 'cdf':
                            result = dist.cdf(k)
                            print(f"P(X ≤ {k}) = {result:.6f}")
                        elif calc_type_poisson.value == 'sf':
                            result = dist.sf(k-1)
                            print(f"P(X ≥ {k}) = {result:.6f}")
                        elif calc_type_poisson.value == 'gt':
                            result = dist.sf(k)
                            print(f"P(X > {k}) = {result:.6f}")
                        elif calc_type_poisson.value == 'lt':
                            result = dist.cdf(k-1) if k > 0 else 0
                            print(f"P(X < {k}) = {result:.6f}")
                        
                        print(f"\nEsperança E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        # Verificar aproximação
                        print("\n" + "="*50)
                        print("🔔 AVISOS DE APROXIMAÇÃO:")
                        
                        if lam > 10:
                            print(f"\n✅ Pode aproximar para NORMAL com μ = {lam:.4f}, σ = {np.sqrt(lam):.4f}")
                            print(f"   Critério: λ = {lam:.4f} > 10")
                        else:
                            print(f"\n⚠️  λ = {lam:.4f} ≤ 10. Use a distribuição de Poisson.")
                        
                        if show_graph_poisson.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(0, int(lam + 5*np.sqrt(lam)))
                            pmf_vals = dist.pmf(x_vals)
                            
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color='coral', edgecolor='black')
                            ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                            ax.axvline(lam, color='green', linestyle='--', linewidth=2, label=f'λ = {lam:.2f}')
                            
                            ax.set_xlabel('Número de Eventos (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição de Poisson: λ={lam}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    prob = prob_widget_poisson.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    if calc_type_poisson_inv.value == 'cdf':
                        k = dist.ppf(prob)
                        print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                        print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                    else:  # sf
                        k = dist.isf(prob)
                        print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                        print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        if int(k) >= 1:
                            print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
                    
                    print(f"\nEsperança E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    print(f"Desvio Padrão σ = {dist.std():.4f}")
                    
                    if show_graph_poisson.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.arange(0, int(lam + 5*np.sqrt(lam)))
                        pmf_vals = dist.pmf(x_vals)
                        
                        ax.bar(x_vals, pmf_vals, alpha=0.7, color='coral', edgecolor='black')
                        ax.axvline(int(k), color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                        ax.axvline(lam, color='green', linestyle='--', linewidth=2, label=f'λ = {lam:.2f}')
                        
                        ax.set_xlabel('Número de Eventos (k)', fontsize=12)
                        ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                        ax.set_title(f'Distribuição de Poisson: λ={lam}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_poisson = False

def atualizar_interface_poisson(change=None):
    global _updating_poisson
    if _updating_poisson:
        return
    _updating_poisson = True
    
    try:
        if modo_poisson.value == 'Direto (k → P)':
            if calc_type_poisson.value == 'interval':
                input_container_poisson.children = [calc_type_poisson, a_poisson, b_poisson]
            else:
                input_container_poisson.children = [calc_type_poisson, k_poisson]
        else:
            input_container_poisson.children = [calc_type_poisson_inv, prob_widget_poisson]
        calcular_poisson()
    finally:
        _updating_poisson = False

# Registrar observadores apenas uma vez
if not hasattr(modo_poisson, '_handler_registered'):
    modo_poisson.observe(atualizar_interface_poisson, 'value')
    calc_type_poisson.observe(atualizar_interface_poisson, 'value')
    lambda_widget.observe(calcular_poisson, 'value')
    k_poisson.observe(calcular_poisson, 'value')
    a_poisson.observe(calcular_poisson, 'value')
    b_poisson.observe(calcular_poisson, 'value')
    prob_widget_poisson.observe(calcular_poisson, 'value')
    calc_type_poisson_inv.observe(calcular_poisson, 'value')
    show_graph_poisson.observe(calcular_poisson, 'value')
    
    modo_poisson._handler_registered = True
    calc_type_poisson._handler_registered = True
    lambda_widget._handler_registered = True
    k_poisson._handler_registered = True
    a_poisson._handler_registered = True
    b_poisson._handler_registered = True
    prob_widget_poisson._handler_registered = True
    calc_type_poisson_inv._handler_registered = True
    show_graph_poisson._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Poisson</h3>"),
    modo_poisson,
    lambda_widget, 
    input_container_poisson,
    show_graph_poisson, 
    output_poisson
]))

atualizar_interface_poisson()

## 3. Distribuição Geométrica

A distribuição geométrica modela o número de tentativas até o primeiro sucesso.

**Parâmetros:**
- p: probabilidade de sucesso
- X: número de tentativas até o primeiro sucesso

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_geom = False

# Criar widgets
modo_geom = widgets.ToggleButtons(
    options=['Direto (k → P)', 'Inverso (P → k)'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

p_geom = widgets.FloatText(value=0.3, description='p (probabilidade):', style={'description_width': 'initial'})
k_geom = widgets.IntText(value=5, description='k (tentativas):', style={'description_width': 'initial'})
a_widget_geom = widgets.IntText(value=2, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget_geom = widgets.IntText(value=8, description='b (limite sup.):', style={'description_width': 'initial'})
prob_widget_geom = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_geom = widgets.Dropdown(
    options=[
        ('P(X = k)', 'pmf'), 
        ('P(X ≤ k)', 'cdf'), 
        ('P(X ≥ k)', 'sf'), 
        ('P(X > k)', 'gt'),
        ('P(a ≤ X ≤ b)', 'interval')
    ],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_geom_inv = widgets.Dropdown(
    options=[('P(X ≤ k) ≥ p', 'cdf'), ('P(X ≥ k) ≤ p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

show_graph_geom = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_geom = widgets.Output()
input_container_geom = widgets.VBox()

def calcular_geometrica(change=None):
    global _updating_geom
    if _updating_geom:
        return
    _updating_geom = True
    
    try:
        with output_geom:
            clear_output(wait=True)
            try:
                p = p_geom.value
                
                if p <= 0 or p > 1:
                    print("❌ Valores inválidos! 0 < p ≤ 1")
                    return
                
                dist = stats.geom(p)
                
                # Modo Direto
                if modo_geom.value == 'Direto (k → P)':
                    if calc_type_geom.value == 'interval':
                        a = a_widget_geom.value
                        b = b_widget_geom.value
                        
                        if a < 1 or a > b:
                            print("❌ Valores inválidos! 1 ≤ a ≤ b")
                            return
                        
                        prob = dist.cdf(b) - (dist.cdf(a-1) if a > 1 else 0)
                        print(f"P({a} ≤ X ≤ {b}) = {prob:.6f}")
                        print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                        print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 1 else 0:.6f}")
                        
                        if show_graph_geom.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(1, min(50, int(dist.mean() + 5*dist.std())))
                            pmf_vals = dist.pmf(x_vals)
                            
                            colors = ['red' if a <= x <= b else 'mediumseagreen' for x in x_vals]
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                            ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                            ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Tentativas (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Geométrica: p={p}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        k = k_geom.value
                        if k < 1:
                            print("❌ Valor inválido! k ≥ 1")
                            return
                        
                        if calc_type_geom.value == 'pmf':
                            result = dist.pmf(k)
                            print(f"P(X = {k}) = {result:.6f}")
                        elif calc_type_geom.value == 'cdf':
                            result = dist.cdf(k)
                            print(f"P(X ≤ {k}) = {result:.6f}")
                        elif calc_type_geom.value == 'sf':
                            result = dist.sf(k-1)
                            print(f"P(X ≥ {k}) = {result:.6f}")
                        elif calc_type_geom.value == 'gt':
                            result = dist.sf(k)
                            print(f"P(X > {k}) = {result:.6f}")
                        
                        print(f"\nEsperança E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        if show_graph_geom.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(1, min(50, int(dist.mean() + 5*dist.std())))
                            pmf_vals = dist.pmf(x_vals)
                            
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color='mediumseagreen', edgecolor='black')
                            ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                            ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Tentativas (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Geométrica: p={p}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    prob = prob_widget_geom.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    if calc_type_geom_inv.value == 'cdf':
                        k = dist.ppf(prob)
                        print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                        print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                    else:  # sf
                        k = dist.isf(prob)
                        print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                        print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        if int(k) >= 1:
                            print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
                    
                    print(f"\nEsperança E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    print(f"Desvio Padrão σ = {dist.std():.4f}")
                    
                    if show_graph_geom.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.arange(1, min(50, int(dist.mean() + 5*dist.std())))
                        pmf_vals = dist.pmf(x_vals)
                        
                        ax.bar(x_vals, pmf_vals, alpha=0.7, color='mediumseagreen', edgecolor='black')
                        ax.axvline(int(k), color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                        ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        ax.set_xlabel('Número de Tentativas (k)', fontsize=12)
                        ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                        ax.set_title(f'Distribuição Geométrica: p={p}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_geom = False

def atualizar_interface_geom(change=None):
    global _updating_geom
    if _updating_geom:
        return
    _updating_geom = True
    
    try:
        if modo_geom.value == 'Direto (k → P)':
            if calc_type_geom.value == 'interval':
                input_container_geom.children = [calc_type_geom, a_widget_geom, b_widget_geom]
            else:
                input_container_geom.children = [calc_type_geom, k_geom]
        else:
            input_container_geom.children = [calc_type_geom_inv, prob_widget_geom]
        calcular_geometrica()
    finally:
        _updating_geom = False

# Registrar observadores apenas uma vez
if not hasattr(modo_geom, '_handler_registered'):
    modo_geom.observe(atualizar_interface_geom, 'value')
    calc_type_geom.observe(atualizar_interface_geom, 'value')
    p_geom.observe(calcular_geometrica, 'value')
    k_geom.observe(calcular_geometrica, 'value')
    a_widget_geom.observe(calcular_geometrica, 'value')
    b_widget_geom.observe(calcular_geometrica, 'value')
    prob_widget_geom.observe(calcular_geometrica, 'value')
    calc_type_geom_inv.observe(calcular_geometrica, 'value')
    show_graph_geom.observe(calcular_geometrica, 'value')
    
    modo_geom._handler_registered = True
    calc_type_geom._handler_registered = True
    p_geom._handler_registered = True
    k_geom._handler_registered = True
    a_widget_geom._handler_registered = True
    b_widget_geom._handler_registered = True
    prob_widget_geom._handler_registered = True
    calc_type_geom_inv._handler_registered = True
    show_graph_geom._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Geométrica</h3>"),
    modo_geom,
    p_geom, 
    input_container_geom,
    show_graph_geom, 
    output_geom
]))

atualizar_interface_geom()

## 4. Distribuição Hipergeométrica

A distribuição hipergeométrica modela o número de sucessos em uma amostra sem reposição.

**Parâmetros:**
- M: tamanho da população
- K: número de sucessos na população
- N: tamanho da amostra
- X: número de sucessos na amostra

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_hyper = False

# Criar widgets
modo_hyper = widgets.ToggleButtons(
    options=['Direto (k → P)', 'Inverso (P → k)'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

M_hyper = widgets.IntText(value=100, description='M (população):', style={'description_width': 'initial'})
K_hyper = widgets.IntText(value=30, description='K (sucessos pop.):', style={'description_width': 'initial'})
N_hyper = widgets.IntText(value=20, description='N (amostra):', style={'description_width': 'initial'})
k_hyper = widgets.IntText(value=8, description='k (sucessos):', style={'description_width': 'initial'})
a_widget_hyper = widgets.IntText(value=4, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget_hyper = widgets.IntText(value=12, description='b (limite sup.):', style={'description_width': 'initial'})
prob_widget_hyper = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_hyper = widgets.Dropdown(
    options=[
        ('P(X = k)', 'pmf'), 
        ('P(X ≤ k)', 'cdf'), 
        ('P(X ≥ k)', 'sf'),
        ('P(a ≤ X ≤ b)', 'interval')
    ],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_hyper_inv = widgets.Dropdown(
    options=[('P(X ≤ k) ≥ p', 'cdf'), ('P(X ≥ k) ≤ p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

show_graph_hyper = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_hyper = widgets.Output()
input_container_hyper = widgets.VBox()

def calcular_hipergeometrica(change=None):
    global _updating_hyper
    if _updating_hyper:
        return
    _updating_hyper = True
    
    try:
        with output_hyper:
            clear_output(wait=True)
            try:
                M = M_hyper.value
                K = K_hyper.value
                N = N_hyper.value
                
                if M < 0 or K < 0 or N < 0 or K > M or N > M:
                    print("❌ Valores inválidos! K ≤ M, N ≤ M, k ≥ 0")
                    return
                
                dist = stats.hypergeom(M, K, N)
                
                # Modo Direto
                if modo_hyper.value == 'Direto (k → P)':
                    if calc_type_hyper.value == 'interval':
                        a = a_widget_hyper.value
                        b = b_widget_hyper.value
                        
                        min_k = max(0, N - M + K)
                        max_k = min(K, N)
                        
                        if a < min_k or b > max_k or a > b:
                            print(f"❌ Valores inválidos! {min_k} ≤ a ≤ b ≤ {max_k}")
                            return
                        
                        prob = dist.cdf(b) - (dist.cdf(a-1) if a > min_k else 0)
                        print(f"P({a} ≤ X ≤ {b}) = {prob:.6f}")
                        print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                        print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > min_k else 0:.6f}")
                        
                        if show_graph_hyper.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(max(0, N-M+K), min(K, N)+1)
                            pmf_vals = dist.pmf(x_vals)
                            
                            colors = ['red' if a <= x <= b else 'mediumpurple' for x in x_vals]
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                            ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Sucessos na Amostra (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Hipergeométrica: M={M}, K={K}, N={N}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        k = k_hyper.value
                        
                        if calc_type_hyper.value == 'pmf':
                            result = dist.pmf(k)
                            print(f"P(X = {k}) = {result:.6f}")
                        elif calc_type_hyper.value == 'cdf':
                            result = dist.cdf(k)
                            print(f"P(X ≤ {k}) = {result:.6f}")
                        elif calc_type_hyper.value == 'sf':
                            result = dist.sf(k-1)
                            print(f"P(X ≥ {k}) = {result:.6f}")
                        
                        print(f"\nEsperança E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        # Verificar aproximação
                        print("\n" + "="*50)
                        print("🔔 AVISOS DE APROXIMAÇÃO:")
                        
                        if M >= 10*N:
                            p_approx = K / M
                            print(f"\n✅ Pode aproximar para BINOMIAL com n={N}, p={p_approx:.4f}")
                            print(f"   Critério: M={M} ≥ 10N={10*N}")
                            print(f"   (População grande o suficiente para considerar independência)")
                        else:
                            print(f"\n⚠️  M={M} < 10N={10*N}. Use a distribuição Hipergeométrica.")
                        
                        if show_graph_hyper.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(max(0, N-M+K), min(K, N)+1)
                            pmf_vals = dist.pmf(x_vals)
                            
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color='mediumpurple', edgecolor='black')
                            ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Sucessos na Amostra (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Hipergeométrica: M={M}, K={K}, N={N}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    prob = prob_widget_hyper.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    if calc_type_hyper_inv.value == 'cdf':
                        k = dist.ppf(prob)
                        print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                        print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                    else:  # sf
                        k = dist.isf(prob)
                        print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                        print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        if int(k) >= 1:
                            print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
                    
                    print(f"\nEsperança E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    print(f"Desvio Padrão σ = {dist.std():.4f}")
                    
                    if show_graph_hyper.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.arange(max(0, N-M+K), min(K, N)+1)
                        pmf_vals = dist.pmf(x_vals)
                        
                        ax.bar(x_vals, pmf_vals, alpha=0.7, color='mediumpurple', edgecolor='black')
                        ax.axvline(int(k), color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                        ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        ax.set_xlabel('Número de Sucessos na Amostra (k)', fontsize=12)
                        ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                        ax.set_title(f'Distribuição Hipergeométrica: M={M}, K={K}, N={N}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_hyper = False

def atualizar_interface_hyper(change=None):
    global _updating_hyper
    if _updating_hyper:
        return
    _updating_hyper = True
    
    try:
        if modo_hyper.value == 'Direto (k → P)':
            if calc_type_hyper.value == 'interval':
                input_container_hyper.children = [calc_type_hyper, a_widget_hyper, b_widget_hyper]
            else:
                input_container_hyper.children = [calc_type_hyper, k_hyper]
        else:
            input_container_hyper.children = [calc_type_hyper_inv, prob_widget_hyper]
        calcular_hipergeometrica()
    finally:
        _updating_hyper = False

# Registrar observadores apenas uma vez
if not hasattr(modo_hyper, '_handler_registered'):
    modo_hyper.observe(atualizar_interface_hyper, 'value')
    calc_type_hyper.observe(atualizar_interface_hyper, 'value')
    M_hyper.observe(calcular_hipergeometrica, 'value')
    K_hyper.observe(calcular_hipergeometrica, 'value')
    N_hyper.observe(calcular_hipergeometrica, 'value')
    k_hyper.observe(calcular_hipergeometrica, 'value')
    a_widget_hyper.observe(calcular_hipergeometrica, 'value')
    b_widget_hyper.observe(calcular_hipergeometrica, 'value')
    prob_widget_hyper.observe(calcular_hipergeometrica, 'value')
    calc_type_hyper_inv.observe(calcular_hipergeometrica, 'value')
    show_graph_hyper.observe(calcular_hipergeometrica, 'value')
    
    modo_hyper._handler_registered = True
    calc_type_hyper._handler_registered = True
    M_hyper._handler_registered = True
    K_hyper._handler_registered = True
    N_hyper._handler_registered = True
    k_hyper._handler_registered = True
    a_widget_hyper._handler_registered = True
    b_widget_hyper._handler_registered = True
    prob_widget_hyper._handler_registered = True
    calc_type_hyper_inv._handler_registered = True
    show_graph_hyper._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Hipergeométrica</h3>"),
    modo_hyper,
    M_hyper, 
    K_hyper, 
    N_hyper, 
    input_container_hyper,
    show_graph_hyper, 
    output_hyper
]))

atualizar_interface_hyper()

## 5. Distribuição Binomial Negativa

A distribuição binomial negativa modela o número de falhas antes de obter r sucessos.

**Parâmetros:**
- r: número de sucessos desejados
- p: probabilidade de sucesso
- X: número de falhas

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_nbinom = False

# Criar widgets
modo_nbinom = widgets.ToggleButtons(
    options=['Direto (k → P)', 'Inverso (P → k)'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

r_nbinom = widgets.IntText(value=5, description='r (sucessos):', style={'description_width': 'initial'})
p_nbinom = widgets.FloatText(value=0.4, description='p (probabilidade):', style={'description_width': 'initial'})
k_nbinom = widgets.IntText(value=7, description='k (falhas):', style={'description_width': 'initial'})
a_widget_nbinom = widgets.IntText(value=3, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget_nbinom = widgets.IntText(value=12, description='b (limite sup.):', style={'description_width': 'initial'})
prob_widget_nbinom = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_nbinom = widgets.Dropdown(
    options=[
        ('P(X = k)', 'pmf'), 
        ('P(X ≤ k)', 'cdf'), 
        ('P(X ≥ k)', 'sf'),
        ('P(a ≤ X ≤ b)', 'interval')
    ],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_nbinom_inv = widgets.Dropdown(
    options=[('P(X ≤ k) ≥ p', 'cdf'), ('P(X ≥ k) ≤ p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

show_graph_nbinom = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_nbinom = widgets.Output()
input_container_nbinom = widgets.VBox()

def calcular_nbinom(change=None):
    global _updating_nbinom
    if _updating_nbinom:
        return
    _updating_nbinom = True
    
    try:
        with output_nbinom:
            clear_output(wait=True)
            try:
                r = r_nbinom.value
                p = p_nbinom.value
                
                if r < 1 or p <= 0 or p >= 1:
                    print("❌ Valores inválidos! r ≥ 1, 0 < p < 1, k ≥ 0")
                    return
                
                dist = stats.nbinom(r, p)
                
                # Modo Direto
                if modo_nbinom.value == 'Direto (k → P)':
                    if calc_type_nbinom.value == 'interval':
                        a = a_widget_nbinom.value
                        b = b_widget_nbinom.value
                        
                        if a < 0 or a > b:
                            print("❌ Valores inválidos! 0 ≤ a ≤ b")
                            return
                        
                        prob = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                        print(f"P({a} ≤ X ≤ {b}) = {prob:.6f}")
                        print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                        print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 0 else 0:.6f}")
                        
                        if show_graph_nbinom.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(0, int(dist.mean() + 5*dist.std()))
                            pmf_vals = dist.pmf(x_vals)
                            
                            colors = ['red' if a <= x <= b else 'goldenrod' for x in x_vals]
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                            ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Falhas (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Binomial Negativa: r={r}, p={p}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        k = k_nbinom.value
                        if k < 0:
                            print("❌ Valor inválido! k ≥ 0")
                            return
                        
                        if calc_type_nbinom.value == 'pmf':
                            result = dist.pmf(k)
                            print(f"P(X = {k}) = {result:.6f}")
                        elif calc_type_nbinom.value == 'cdf':
                            result = dist.cdf(k)
                            print(f"P(X ≤ {k}) = {result:.6f}")
                        elif calc_type_nbinom.value == 'sf':
                            result = dist.sf(k-1)
                            print(f"P(X ≥ {k}) = {result:.6f}")
                        
                        print(f"\nEsperança E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        if show_graph_nbinom.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.arange(0, int(dist.mean() + 5*dist.std()))
                            pmf_vals = dist.pmf(x_vals)
                            
                            ax.bar(x_vals, pmf_vals, alpha=0.7, color='goldenrod', edgecolor='black')
                            ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            ax.set_xlabel('Número de Falhas (k)', fontsize=12)
                            ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                            ax.set_title(f'Distribuição Binomial Negativa: r={r}, p={p}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    prob = prob_widget_nbinom.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    if calc_type_nbinom_inv.value == 'cdf':
                        k = dist.ppf(prob)
                        print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                        print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                    else:  # sf
                        k = dist.isf(prob)
                        print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                        print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                        print(f"k = {int(k)}")
                        if int(k) >= 1:
                            print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
                    
                    print(f"\nEsperança E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    print(f"Desvio Padrão σ = {dist.std():.4f}")
                    
                    if show_graph_nbinom.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.arange(0, int(dist.mean() + 5*dist.std()))
                        pmf_vals = dist.pmf(x_vals)
                        
                        ax.bar(x_vals, pmf_vals, alpha=0.7, color='goldenrod', edgecolor='black')
                        ax.axvline(int(k), color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                        ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        ax.set_xlabel('Número de Falhas (k)', fontsize=12)
                        ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                        ax.set_title(f'Distribuição Binomial Negativa: r={r}, p={p}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_nbinom = False

def atualizar_interface_nbinom(change=None):
    global _updating_nbinom
    if _updating_nbinom:
        return
    _updating_nbinom = True
    
    try:
        if modo_nbinom.value == 'Direto (k → P)':
            if calc_type_nbinom.value == 'interval':
                input_container_nbinom.children = [calc_type_nbinom, a_widget_nbinom, b_widget_nbinom]
            else:
                input_container_nbinom.children = [calc_type_nbinom, k_nbinom]
        else:
            input_container_nbinom.children = [calc_type_nbinom_inv, prob_widget_nbinom]
        calcular_nbinom()
    finally:
        _updating_nbinom = False

# Registrar observadores apenas uma vez
if not hasattr(modo_nbinom, '_handler_registered'):
    modo_nbinom.observe(atualizar_interface_nbinom, 'value')
    calc_type_nbinom.observe(atualizar_interface_nbinom, 'value')
    r_nbinom.observe(calcular_nbinom, 'value')
    p_nbinom.observe(calcular_nbinom, 'value')
    k_nbinom.observe(calcular_nbinom, 'value')
    a_widget_nbinom.observe(calcular_nbinom, 'value')
    b_widget_nbinom.observe(calcular_nbinom, 'value')
    prob_widget_nbinom.observe(calcular_nbinom, 'value')
    calc_type_nbinom_inv.observe(calcular_nbinom, 'value')
    show_graph_nbinom.observe(calcular_nbinom, 'value')
    
    modo_nbinom._handler_registered = True
    calc_type_nbinom._handler_registered = True
    r_nbinom._handler_registered = True
    p_nbinom._handler_registered = True
    k_nbinom._handler_registered = True
    a_widget_nbinom._handler_registered = True
    b_widget_nbinom._handler_registered = True
    prob_widget_nbinom._handler_registered = True
    calc_type_nbinom_inv._handler_registered = True
    show_graph_nbinom._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Binomial Negativa</h3>"),
    modo_nbinom,
    r_nbinom, 
    p_nbinom, 
    input_container_nbinom,
    show_graph_nbinom, 
    output_nbinom
]))

atualizar_interface_nbinom()